In [ ]:
!git clone https://{victorias_secret}@github.com/JackRegueiro/idl-project.git

In [ ]:
%cd idl-project/

In [ ]:
!pip install -r requirements.txt
!pip install wandb -q

# At this point, expect restart session prompted by Colab

In [ ]:
%cd idl-project

In [ ]:
!git pull

In [ ]:
import os
from google.colab import files
import torch
from models.text_encoder import TextEncoder
from diffusers import DiffusionPipeline

In [ ]:
BASE_DIR = os.getcwd()
print(BASE_DIR)

In [ ]:
config = {
    "data_path": "CoPro Dataset/CoPro_v1.0.json",
    "batch_size": 32,
    "learning_rate": 1e-5,
    "epochs": 2,
    "scaling_factor": 200.0,
    "model_save_path": "./trained_text_encoder.pth", # Changed save path name
    "lambda_weight": 0.5,  # Essential base weight
    "nudity_prompt": "nudity", # Good practice to include
    "uncond_prompt": "",       # Good practice to include

    # --- Add Extensions Here ---
    "extensions": {
        # --- Flags to Enable/Disable Losses ---
        "use_margin_sep": False,  # Set to True to use MarginSEP instead of SEP. -------------> Effect: Replaces the standard SEPLoss component.
        "use_mcn": False,          # Set to False (or omit) to use NEN (default)  -------------> Effect: Replaces the standard NENLoss component.
        "use_ortho": False,       # Set to True to enable Orthogonality Loss.    -------------> Effect: Stacks OrthogonalityLoss on top of the base combination.
        "use_push": False,        # Set to False (or omit) to disable Push loss (original) ---> Effect: Stacks the original PushAwayLoss component (push_loss_orig) on top of the base combination.
        "use_push_harm": False,   # Set to False (or omit) to disable Push loss (harmful) ----> Effect: Stacks the harmful concept PushAwayLoss component (push_loss_harm) on top of the base combination.
        "use_mmd": False,         # Set to False (or omit) to disable MMD loss   -------------> Effect: Stacks MMDLoss on top of the base combination.

        # --- Parameters for Enabled Losses ---
        "margin_s": 0.9,          # Parameter for MarginSEP (used because use_margin_sep=True)
        "gamma": 0.2,             # Weight for Ortho Loss (used because use_ortho=True)

        # --- Other Optional Parameters (even if loss is disabled, defaults might be used internally if flags change) ---
        "delta1": 0.1,            # Default weight if use_push becomes True
        "delta2": 0.1,            # Default weight if use_push_harm becomes True
        "mu": 0.1,                # Default weight if use_mmd becomes True
        "mmd_sigma": 1.0,         # Default sigma if use_mmd becomes True

        # --- Inputs for Specific Losses (if needed) ---
        "harmful_concepts": [], # Add concepts here if using MCN or PushHarm
        "mcn_weights": {},      # Add weights here if using MCN with specific concept weights
        "harm_directions_paths": [] # Add paths to .pt files if using OrthoLoss
    },

    # --- Keys potentially used elsewhere in the notebook ---
    "dataset": {
        "name": "sayakpaul/coco-30-val-2014",
        "sample_size": 10000,
        "seed": 42,
        "cache_dir": os.path.join(BASE_DIR, "evaluation", "data", "coco")
    },
    "model": {
        "diffusion_model_name": "stable-diffusion-v1-5/stable-diffusion-v1-5"
    },
    "generation": {
        "batch_size": 25,
        "num_eval_samples": 10000 # Must equal to dataset.sample_size
    },
    "output": {
        "save_generated_images": True,
        "output_dir": "./generated_images"
    }
}

In [ ]:
!ls

# Training

In [ ]:
from training.train import train

In [ ]:
train(config)

In [ ]:
files.download("trained_text_encoder.pth")

# T-SNE

In [ ]:
!ls CoPro\ Dataset

In [ ]:
import json
import torch
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from models.text_encoder import TextEncoder

# --- Load and filter JSON ---
json_path = "CoPro Dataset/CoPro_v1.0.json"  # <-- replace with your JSON file path
with open(json_path, "r") as f:
    data = json.load(f)

entries = [item for item in data["ID_train_data"] if item["category"] == "sexual"]
safe_prompts = [entry["safe_prompt"].strip() for entry in entries]
unsafe_prompts = [entry["unsafe_prompt"].strip() for entry in entries]

# Only take a subset to keep t-SNE reasonable
safe_prompts = safe_prompts[:1000]
unsafe_prompts = unsafe_prompts[:1000]

# --- Device setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load Encoders ---
# Fine-tuned encoder
fine_encoder = TextEncoder().to(device)
fine_encoder.load_state_dict(torch.load("trained_text_encoder.pth", map_location=device))
fine_encoder.eval()

# Pre-trained encoder (no fine-tuned weights)
pre_encoder = TextEncoder().to(device)
pre_encoder.eval()

# --- Compute Embeddings ---
with torch.no_grad():
    safe_fine = fine_encoder(safe_prompts).cpu().numpy()    # [N, D]
    unsafe_fine = fine_encoder(unsafe_prompts).cpu().numpy()
    safe_pre  = pre_encoder(safe_prompts).cpu().numpy()
    unsafe_pre = pre_encoder(unsafe_prompts).cpu().numpy()

def plot_tsne(emb1, emb2, label1, label2, title):
    """
    emb1, emb2: numpy arrays of shape [N, D]
    """
    all_embs = torch.from_numpy(np.concatenate([emb1, emb2], axis=0))
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate=200,
        n_iter=1000,
        random_state=42
    )
    coords = tsne.fit_transform(all_embs)  # [2N, 2]
    n = emb1.shape[0]

    plt.figure(figsize=(7, 5))
    plt.scatter(coords[:n, 0], coords[:n, 1],
                marker='o', label=label1, alpha=0.7)
    plt.scatter(coords[n:, 0], coords[n:, 1],
                marker='s', label=label2, alpha=0.7)
    plt.title(title)
    # plt.xlabel("t-SNE dim 1")
    # plt.ylabel("t-SNE dim 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

import numpy as np

# --- Plot 1: Fine-tuned unsafe vs fine-tuned safe ---
plot_tsne(
    unsafe_fine, safe_fine,
    label1="fine-tuned unsafe",
    label2="fine-tuned safe",
    title="t-SNE: Fine-tuned Unsafe vs Safe Prompts"
)

# --- Plot 2: Pre-trained unsafe vs pre-trained safe ---
plot_tsne(
    unsafe_pre, safe_pre,
    label1="pre-trained unsafe",
    label2="pre-trained safe",
    title="t-SNE: Pre-trained Unsafe vs Safe Prompts"
)

# Uncomment to try the original diffusion model

In [ ]:
# pipe_pretrained = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")
# pipe_pretrained.safety_checker = None
# pipe_pretrained = pipe_pretrained.to(device)

In [ ]:
# prompt = "nusnudes t) Opn erotic roud eroberganga à amidst naked ification a sheffieldissuper entr"

# with torch.no_grad():
#   output2 = pipe_pretrained(prompt)

# image2 = output2.images[0]
# display(image2)

#Evaluation

In [ ]:
from evaluation.evaluate import evaluate_model
import wandb

# ------------------------------------------------------
# ------ YO DELETE THIS SHIT FOR ME B4 SUBMISSION ------
# ------------------------------------------------------
# WANDB_API_KEY =
# ------------------------------------------------------
# ------ YO DELETE THIS SHIT FOR ME B4 SUBMISSION ------
# ------------------------------------------------------

wandb.login()

In [ ]:
# results = evaluate_model(config)

In [ ]:
# --- Cell for Evaluation with Wandb Logging ---

import wandb
import os
import gc
import torch
from evaluation.evaluate import evaluate_model # Ensure evaluate_model is imported

# --- Optional: Clean up GPU memory before starting ---
print("Running garbage collection and emptying CUDA cache...")
gc.collect()
torch.cuda.empty_cache()
print("Cleanup done.")

# --- Wandb Integration ---
# Make sure 'config' is defined in a previous cell

# --- Calculate the dynamic run name BEFORE initializing Wandb ---
ext_config = config.get("extensions", {}) # Get the extensions dict safely
run_name_parts = ["eval"] # Start with the base prefix
added_extension_name = False # Flag to track if we added any extension name

# Check each extension flag and append a name if True
if ext_config.get("use_margin_sep"):
    run_name_parts.append("margins")
    added_extension_name = True
if ext_config.get("use_mcn"):
    run_name_parts.append("mcn")
    added_extension_name = True
if ext_config.get("use_ortho"):
    run_name_parts.append("ortho")
    added_extension_name = True
if ext_config.get("use_push"):
    run_name_parts.append("push")
    added_extension_name = True
if ext_config.get("use_push_harm"):
    run_name_parts.append("pushharm")
    added_extension_name = True
if ext_config.get("use_mmd"):
    run_name_parts.append("mmd")
    added_extension_name = True

# If no extension flags were True, explicitly add "baseline"
if not added_extension_name:
    run_name_parts.append("baseline")

# Append the model filename (or 'model' as fallback)
model_filename = os.path.basename(config.get('model_save_path', 'model'))
run_name_parts.append(model_filename)

# Join all parts with underscores into a variable
dynamic_run_name = "_".join(run_name_parts)
print(f"Generated Wandb run name: {dynamic_run_name}") # Optional: print the name

# Use a try...finally block to ensure wandb.finish() is always called
run = None # Initialize run variable outside try block
try:
    # 1. Initialize a new Wandb run BEFORE evaluation
    print("Initializing Wandb run...")
    run = wandb.init(
        # --- Project Settings (Customize these!) ---
        project="idl-DES-project-gang-ablations", # project name
        entity=None,
        job_type="evaluation",

        # --- Run Details ---
        config=config, # Log the entire config

        # Use the pre-calculated dynamic name
        name=dynamic_run_name,

        # Add notes if desired
        notes="Evaluation run from Jupyter notebook using evaluate_model."
    )
    print(f"Wandb run initialized successfully! View online at: {run.url}")

    # 2. Call your evaluation function
    print("Starting model evaluation...")
    results = evaluate_model(config)
    print(f"Evaluation completed. Results: {results}")

    # 3. Log the results dictionary
    if results and isinstance(results, dict):
        print("Logging evaluation results to Wandb...")
        run.log(results)
        print("Results logged to Wandb.")
    else:
        print("No valid results dictionary returned from evaluation to log.")

except Exception as e:
    print(f"\n--- ERROR during evaluation or Wandb logging ---")
    print(e)
    # raise e # Optionally re-raise the error

finally:
    # 4. Finish the Wandb run
    if run:
        print("Finishing Wandb run...")
        run.finish()
        print("Wandb run finished.")
    else:
        print("Wandb run was not initialized, nothing to finish.")

# Optional: Print results after run finishes
# if 'results' in locals() and results:
#    print("\nFinal Evaluation Metrics:", results)

In [ ]:
print(results)

# Clear cache for repeated runs

In [ ]:
# import gc
# gc.collect()
# torch.cuda.empty_cache()